# Security Data — Logistic Regression (Multiple Logistic Regression)

## Dataset
Data sourced from `security_data_10000.csv` (synthetic data generated from original class dataset).

| Column | Description |
|--------|-------------|
| Sector | Industry sector of the company |
| CEO_Gender | Gender of the CEO (Male/Female) |
| Size | Company size (Large, Medium, Small) |
| Security_Invest | Security investment amount |
| Security_Breach_Att | Number of security breach attempts |
| Succ_Sec_Breaches | Number of successful security breaches |
| Sec_Rating | Overall security rating (High/Medium/Low) |
| CEO_Sec_Exp | CEO's level of security experience (High/Medium/Low) |
| LOT_in_Business | Length of time in business (years) |
| Stock_Market | Whether the company is listed on the stock market (Yes/No) |

---

## Objective
Predict whether a company's CEO is **Male or Female** using all available features.

**Target variable:** `CEO_Gender` (Male = True / Female = False)

---

## Data Analysis Plan

1. **Load Data** — Read CSV into DataFrame
2. **Train-Test Split** — Split before any feature engineering
3. **Create Binary Label** — Encode `CEO_Gender` as True/False
4. **Drop Target from Features** — Remove `CEO_Gender` from input features
5. **Encode Categorical Features** — One-hot encode remaining categorical columns
6. **Fit Logistic Regression** — Use `class_weight='balanced'` to handle class imbalance
7. **Evaluate** — Confusion matrix and classification report on train and test

## Step 1 — Load Dataset

In [34]:
#LOAD THE DATASET


import pandas as pd

security_df = pd.read_csv("https://raw.githubusercontent.com/eddiejaques/ml-code-samples/refs/heads/main/Security%20Breaches%20Dataset/security-data.csv")

security_df.head()

,Sector,CEO_Gender,Size,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,Sec_Rating,CEO_Sec_Exp,LOT_in_Business,Stock_Market
0,Banking,Female,Small,17,11,2,Medium,High,13,No
1,Banking,Male,Small,18,12,4,High,Low,9,No
2,Banking,Male,Small,17,12,4,High,Medium,22,No
3,Banking,Male,Small,24,13,1,High,Medium,3,Yes
4,Banking,Male,Small,32,14,3,High,Medium,4,Yes


## Step 2 — Train-Test Split
Split performed **before** any feature engineering to prevent data leakage.

In [35]:
from sklearn.model_selection import train_test_split
security_df_train, security_df_test = train_test_split(security_df, test_size=0.2, random_state=42)


### Step 3 — Exploratory Data Analysis (EDA)

In [36]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,48.000000,48.000000,48.000000,48.000000
mean,66.604167,54.979167,24.229167,9.125000
std,75.107225,59.456220,24.295381,6.625627
min,12.000000,7.000000,0.000000,1.000000
25%,21.000000,16.250000,4.000000,3.000000
50%,32.000000,40.000000,19.000000,8.000000
75%,93.000000,77.000000,34.000000,14.000000
max,435.000000,321.000000,100.000000,22.000000


In [48]:
security_df_train.isna().sum()

,0
Sector,0
Size,0
Security_Invest,0
Security_Breach_Att,0
Succ_Sec_Breaches,0
Sec_Rating,0
CEO_Sec_Exp,0
LOT_in_Business,0
Stock_Market,0


## Step 4 — Create Binary Label
Encode `CEO_Gender` as a boolean: `True` = Male, `False` = Female.
This will be the target variable for the logistic regression model.

In [37]:
security_df_train_label = (security_df_train["Stock_Market"] == 'Male')

security_df_test_label = (security_df_test["Stock_Market"] =='Male')

In [38]:
security_df_train.shape

(48, 10)

## Step 5 — Drop Target Column from Features
`CEO_Gender` must be removed from the input features — it is the target variable, not a predictor.

In [39]:
security_df_train = security_df_train.drop("Stock_Market", axis=1)
security_df_train.shape

(48, 9)

In [40]:
security_df_test = security_df_test.drop("Stock_Market", axis=1)
security_df_test.shape

(12, 9)

In [41]:
security_df_train.describe()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
count,48.000000,48.000000,48.000000,48.000000
mean,66.604167,54.979167,24.229167,9.125000
std,75.107225,59.456220,24.295381,6.625627
min,12.000000,7.000000,0.000000,1.000000
25%,21.000000,16.250000,4.000000,3.000000
50%,32.000000,40.000000,19.000000,8.000000
75%,93.000000,77.000000,34.000000,14.000000
max,435.000000,321.000000,100.000000,22.000000


In [42]:

security_df_train_cat = pd.get_dummies(security_df_train[["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "CEO_Gender"]],dtype=int)

security_df_test_cat = pd.get_dummies(security_df_test[["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "CEO_Gender"]],dtype=int)

security_df_train_final = pd.concat([security_df_train, security_df_train_cat], axis=1)

security_df_test_final = pd.concat([security_df_test, security_df_test_cat], axis=1)

security_df_train_final = security_df_train_final.drop(["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "CEO_Gender"], axis=1)

security_df_test_final = security_df_test_final.drop(["Sector", "Size", "Sec_Rating", "CEO_Sec_Exp", "CEO_Gender"], axis=1)

## Step 6 — Encode Categorical Features
One-hot encode all remaining categorical columns. `dtype=int` ensures 0/1 output instead of True/False.
Original categorical columns are dropped after encoding to avoid duplication.

In [43]:
security_df_train_final.head()

,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes
31,91,184,85,21,0,0,1,0,1,0,0,0,1,1,0,0,0,1
3,24,13,1,3,1,0,0,0,0,1,1,0,0,0,0,1,0,1
52,87,77,33,8,0,0,1,1,0,0,0,1,0,0,0,1,0,1
17,27,20,8,4,0,1,0,0,0,1,1,0,0,1,0,0,0,1
8,99,77,34,3,0,0,1,0,0,1,0,1,0,0,0,1,0,1


In [44]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

security_df_train_final = pd.DataFrame(imputer.fit_transform(security_df_train_final), columns=security_df_train_final.columns)

security_df_test_final = pd.DataFrame(imputer.transform(security_df_test_final), columns=security_df_test_final.columns)

security_df_train_final.head()


,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes
0,91.0,184.0,85.0,21.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,24.0,13.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,87.0,77.0,33.0,8.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
3,27.0,20.0,8.0,4.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4,99.0,77.0,34.0,3.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


In [45]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numerical_cols = ["Security_Invest", "Security_Breach_Att", "Succ_Sec_Breaches", "LOT_in_Business"]

security_df_train_scaled = pd.DataFrame(scaler.fit_transform(security_df_train_final[numerical_cols]), columns=numerical_cols)

security_df_test_scaled = pd.DataFrame(scaler.transform(security_df_train_final[numerical_cols]), columns=numerical_cols)

security_df_train_final = pd.concat([security_df_train_final.drop(numerical_cols, axis=1),
                                     security_df_train_scaled], axis=1)
security_df_test_final =  pd.concat([security_df_test_final.drop(numerical_cols, axis=1),
                                     security_df_test_scaled], axis=1)

security_df_train_final.head()

,Sector_Banking,Sector_Health Care,Sector_Hospitality,Size_Large,Size_Medium,Size_Small,Sec_Rating_High,Sec_Rating_Low,Sec_Rating_Medium,CEO_Sec_Exp_High,CEO_Sec_Exp_Low,CEO_Sec_Exp_Medium,Stock_Market_No,Stock_Market_Yes,Security_Invest,Security_Breach_Att,Succ_Sec_Breaches,LOT_in_Business
0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.328251,2.192978,2.527803,1.811250
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.573247,-0.713523,-0.966232,-0.934223
2,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.274430,0.374290,0.364829,-0.171592
3,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.532882,-0.594544,-0.675063,-0.781697
4,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.435892,0.374290,0.406424,-0.934223


## Step 7 — Fit Logistic Regression
`class_weight='balanced'` corrects for class imbalance (more Male than Female CEOs in the dataset).
Without this, the model defaults to predicting the majority class only.

In [47]:
from sklearn.linear_model import LogisticRegression

logistic_model = log_reg = LogisticRegression(class_weight="balanced")

logistic_model.fit(security_df_train_final, security_df_train_label)

security_df_train_label_pred = logistic_model.predict(security_df_train_final)

security_df_test_label_pred = logistic_model.predict(security_df_test_final)



ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print("Confusion Matrix - Train:")

print(confusion_matrix(security_df_train_label, security_df_train_label_pred))

print("Classification Report - Train:")

print(classification_report(security_df_train_label, security_df_train_label_pred))

print("Confusion Matrix - Test:")

print(confusion_matrix(security_df_test_label, security_df_test_label_pred))

print("Classification Report - Test:")

print(classification_report(security_df_test_label, security_df_test_label_pred))

## Step 8 — Evaluate
- **Confusion matrix** — shows True/False Positives and Negatives
- **Precision** — of all predicted Male/Female, how many were correct
- **Recall** — of all actual Male/Female, how many were caught
- **F1-score** — harmonic mean of precision and recall

# Summary


We tried